<a href="https://colab.research.google.com/github/nityaprakash-b/Intelligent-Banking-Assistant/blob/main/Bank_Intelligence_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
The goal of this project is to build 'BankGPT', an advanced RAG-based AI system specialized in the Indian banking sector for the 2025-2026 fiscal year. This system will involve building a robust pipeline including structured document generation, hierarchical semantic chunking, hybrid retrieval (combining dense FAISS embeddings and sparse BM25 search), and a Llama 3.2 1B/3B inference engine optimized with 4-bit quantization. The final solution will feature a multi-hop query agent for complex financial reasoning, an automated evaluation suite to ensure factual accuracy, and a Gradio-based interactive dashboard for real-time analysis.

## Phase 1: Environment Setup & Package Management

### Subtask:
Install dependencies and set up the modular directory structure and configuration for BankGPT.


**Reasoning**:
First, I need to install the essential Python packages to support high-performance retrieval and quantized model inference in the Colab environment.



In [ ]:
!pip install -q transformers bitsandbytes accelerate faiss-cpu rank_bm25 sentence-transformers gradio pydantic cross-encoder
print('Libraries installed successfully.')

ERROR: Could not find a version that satisfies the requirement cross-encoder (from versions: none)
ERROR: No matching distribution found for cross-encoder
Libraries installed successfully.


**Reasoning**:
I will fix the pip install command by removing the non-existent 'cross-encoder' package (as it is included in sentence-transformers) and then proceed to create the project directory structure.



In [ ]:
import os

# 1. Re-run installation without the invalid 'cross-encoder' package
!pip install -q transformers bitsandbytes accelerate faiss-cpu rank_bm25 sentence-transformers gradio pydantic

# 2. Define and create the modular directory structure
directories = [
    'data/raw',
    'data/processed',
    'models/embeddings',
    'src/retrieval',
    'src/generation',
    'src/utils',
    'logs'
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    print(f'Created directory: {directory}')

print('\nEnvironment setup and directory structure completed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 83.1 MB/s eta 0:00:00
Created directory: data/raw
Created directory: data/processed
Created directory: models/embeddings
Created directory: src/retrieval
Created directory: src/generation
Created directory: src/utils
Created directory: logs

Environment setup and directory structure completed.


## Phase 2: Structured Document Generation

### Subtask:
Generate the knowledge base of synthetic financial documents for the Indian banking sector (2025-2026) with YAML metadata.


**Reasoning**:
I will create a script to generate and save the synthetic documents to the 'data/raw' directory with appropriate metadata.



In [36]:
import os

os.makedirs('data/raw', exist_ok=True)
docs = [
    {'filename': 'glossary.md', 'content': '---\ntitle: Indian Banking Glossary 2025\ncategory: Reference\ndate: 2025-01-10\nregion: India\n---\n# Indian Banking Glossary\n\n- **Unified Ledger Interface (ULI)**: A cross-institutional platform.\n- **Digital Rupee (e-Rupee)**: CBDC for India.\n'},
    {'filename': 'vikas_bank_report.md', 'content': '---\ntitle: Vikas Bank Annual Report FY25-26\ncategory: Financial Report\ndate: 2025-06-15\nregion: India\n---\n# Vikas Bank Annual Performance\n\nVikas Bank Net NPA has dropped to 2.1% in Q1 FY25-26.\n'},
    {'filename': 'apex_national_bank_fy25.md', 'content': '---\ntitle: Apex National Bank Annual Report 2025\ncategory: Financial Report\ndate: 2025-07-20\nregion: India\n---\n# Apex National Bank Highlights\n\nApex National Bank reported a Net Profit of ₹12,500 crores.\n'}
]

for doc in docs:
    with open(os.path.join('data/raw', doc['filename']), 'w', encoding='utf-8') as f:
        f.write(doc['content'])
print('Synthetic documents restored.')

Synthetic documents restored.


In [ ]:
expanded_docs = [
    {
        'filename': 'apex_national_bank_fy25.md',
        'content': "---\ntitle: Apex National Bank Annual Report 2025\ncategory: Financial Report\ndate: 2025-07-20\nregion: India\n---\n# Apex National Bank: Performance Highlights\n\nApex National Bank has reported a stellar performance for FY 2024-25. Key highlights include:\n- **Net Profit**: \u20b912,500 crores, a 22% YoY increase.\n- **Retail Loan Growth**: Driven by a 30% surge in digital home loan applications via the 'ApexSmart' app.\n- **Micro-Lending**: The bank has disbursed \u20b92,000 crores to SHGs (Self Help Groups) in rural Maharashtra and Karnataka.\n- **Dividend**: The Board has recommended a dividend of \u20b95 per equity share.\n\n## Future Outlook\nThe bank plans to open 500 new 'Phygital' branches by 2026, combining physical presence with digital kiosks. The focus remains on maintaining a healthy CASA ratio of 42%."
    },
    {
        'filename': 'global_india_bank_2026.md',
        'content': "---\ntitle: Global India Bank Strategic Roadmap 2026\ncategory: Strategy\ndate: 2026-01-05\nregion: India\n---\n# Global India Bank: Vision 2026\n\nGlobal India Bank (GIB) aims to become the leader in cross-border trade finance using the ULI framework. \n\n### Key Strategic Pillars:\n1. **Sustainability**: Committing to Net Zero by 2040. GIB will stop financing new coal-based power plants by 2027.\n2. **Wealth Management**: Launching 'GIB-Elite' for High Net Worth Individuals (HNIs) with AI-driven portfolio rebalancing.\n3. **Cybersecurity**: A dedicated budget of \u20b9800 crores for Quantum-Resistant Encryption (QRE) implementation across all core banking servers.\n\n### Financial Metrics:\n- **Current ROA**: 1.8%\n- **Target ROE**: 16.5% by Q4 2026."
    },
    {
        'filename': 'rbi_guidelines_2025_2026.md',
        'content': "---\ntitle: RBI Master Circular - Consumer Interest Rates\ncategory: Regulatory\ndate: 2025-11-12\nregion: India\n---\n# RBI Guidelines on Interest Rates and Consumer Protection\n\n### 1. External Benchmark Lending Rate (EBLR)\nAll floating rate personal or retail loans (housing, auto, etc.) and floating rate loans to MSMEs extended by banks must be benchmarked to an external rate, such as the RBI Repo Rate or Treasury Bill rates.\n\n### 2. Reset of Interest Rates\nBanks must provide borrowers the option to switch to a fixed interest rate at the time of interest rate reset. Any charges for this switch must be disclosed transparently in the Key Fact Statement (KFS).\n\n### 3. Penal Charges\nPenalty for non-compliance of loan contract terms shall be treated as 'penal charges' and not as 'penal interest'. There shall be no capitalization of penal charges."
    },
    {
        'filename': 'customer_faq_2026.md',
        'content': "---\ntitle: Consumer Banking FAQ 2026\ncategory: FAQ\ndate: 2026-02-10\nregion: India\n---\n# Frequently Asked Questions for Indian Bank Customers\n\n**Q: How does the Unified Ledger Interface (ULI) help me?**\nA: ULI allows banks to access your verified financial data instantly (with your consent). This means your personal loan approval can happen in minutes rather than days, as the bank doesn't need physical documents for credit appraisal.\n\n**Q: What is the maximum insurance on my deposits?**\nA: Under the DICGC act, your deposits (savings, fixed, current) are insured up to \u20b95 lakhs per bank.\n\n**Q: Can I change my EMI date?**\nA: Yes, per RBI 2025 guidelines, banks must offer a flexible 'EMI-Date-Reset' option once a year without heavy processing fees, provided the request is made 15 days in advance."
    }
]

for doc in expanded_docs:
    path = os.path.join('data/raw', doc['filename'])
    with open(path, 'w') as f:
        f.write(doc['content'])
    print(f'Saved: {path}')

print('Additional synthetic documents generated.')

Saved: data/raw/apex_national_bank_fy25.md
Saved: data/raw/global_india_bank_2026.md
Saved: data/raw/rbi_guidelines_2025_2026.md
Saved: data/raw/customer_faq_2026.md
Additional synthetic documents generated.


In [ ]:
# Re-generate the PDF to include all documents (approx 10 pages of content)
create_pdf_from_kb('data/raw', 'outputs/BankGPT_Expanded_KB.pdf')

PDF successfully generated at: outputs/BankGPT_Expanded_KB.pdf


In [ ]:
# Pass through Phase 3: Ingestion
!python src/ingestion.py

# Pass through Phase 4: Indexing
!python src/indexer.py

In [ ]:
import shutil
import os

# Define paths to clear
paths_to_clear = ['data/processed', 'models/embeddings']

for path in paths_to_clear:
    if os.path.exists(path):
        shutil.rmtree(path)
        os.makedirs(path)
        print(f'Cleared and recreated: {path}')

print('\nSystem state is now clean. Re-running ingestion and indexing...')


Cleared and recreated: data/processed
Cleared and recreated: models/embeddings

System state is now clean. Re-running ingestion and indexing...


In [ ]:
# Re-run Phase 3: Ingestion with new expanded data
!python src/ingestion.py

# Re-run Phase 4: Indexing with new expanded data
!python src/indexer.py

Successfully created 11 chunks in data/processed/chunks.json
Loading embedding model: BAAI/bge-small-en-v1.5...
Loading weights: 100% 199/199 [00:00<00:00, 5027.08it/s]
Generating embeddings for FAISS...
Batches: 100% 1/1 [00:03<00:00,  3.32s/it]
Building BM25 index...
Indices saved successfully.


### Pipeline Refreshed
The previous embeddings and chunks have been deleted. The system has now processed the expanded knowledge base (~10 pages).

**Ready for your final approval to move to Phase 6 (LLM Generation).**

### Data Expansion Complete
The knowledge base has been expanded, the PDF is updated at `outputs/BankGPT_Expanded_KB.pdf`, and the search indices have been rebuilt. **Awaiting approval to proceed to Phase 6 inference.**

In [ ]:
!pip install -q fpdf

  Preparing metadata (setup.py) ... done


In [ ]:
import os
from fpdf import FPDF

def create_pdf_from_kb(input_dir, output_path):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)

    for filename in sorted(os.listdir(input_dir)):
        if filename.endswith('.md'):
            pdf.add_page()
            pdf.set_font('Arial', 'B', 16)
            pdf.cell(200, 10, txt=f'Document: {filename}', ln=True, align='C')
            pdf.ln(10)

            file_path = os.path.join(input_dir, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()

            pdf.set_font('Arial', '', 12)
            # Multi_cell handles long text and wrapping
            pdf.multi_cell(0, 10, txt=content.encode('latin-1', 'replace').decode('latin-1'))

    pdf.output(output_path)
    print(f'PDF successfully generated at: {output_path}')

# Generate the PDF
os.makedirs('outputs', exist_ok=True)
create_pdf_from_kb('data/raw', 'outputs/BankGPT_KnowledgeBase.pdf')

PDF successfully generated at: outputs/BankGPT_KnowledgeBase.pdf


## Phase 3: Hierarchical Document Chunking & Metadata Enrichment

### Subtask:
Implement the `src/ingestion.py` script to process documents into chunks with injected metadata preambles.

In [ ]:
import os
import yaml
import json

# Updated ingestion logic to correctly stringify date objects before JSON serialization
ingestion_script = """
import os
import yaml
import json
import re
from datetime import date

class BankGPTIngestor:
    def __init__(self, chunk_size=600, overlap=50):
        self.chunk_size = chunk_size
        self.overlap = overlap

    def parse_markdown(self, file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        match = re.match(r'^---\\s+(.*?)\\s+---\\s+(.*)', content, re.DOTALL)
        if match:
            metadata = yaml.safe_load(match.group(1))
            body = match.group(2)
        else:
            metadata = {'filename': os.path.basename(file_path)}
            body = content

        # Ensure all metadata values (especially dates) are JSON serializable
        processed_metadata = {}
        for key, value in metadata.items():
            if isinstance(value, date):
                processed_metadata[key] = value.isoformat()
            else:
                processed_metadata[key] = value

        return processed_metadata, body

    def chunk_text(self, text, metadata):
        chunks = []
        preamble = f"Source: {metadata.get('title', 'Unknown')} | Bank: {metadata.get('bank', 'General')} | Date: {metadata.get('date', '2025')} | "

        start = 0
        while start < len(text):
            end = start + self.chunk_size
            chunk_body = text[start:end]

            chunks.append({
                "text_content": preamble + chunk_body.strip(),
                "metadata_dict": metadata,
                "source_file": metadata.get('filename', 'unknown')
            })
            start += (self.chunk_size - self.overlap)

        return chunks

    def run(self, raw_dir, output_path):
        all_chunks = []
        if not os.path.exists(raw_dir):
            print(f'Error: {raw_dir} not found.')
            return 0

        for filename in os.listdir(raw_dir):
            if filename.endswith('.md'):
                meta, body = self.parse_markdown(os.path.join(raw_dir, filename))
                meta['filename'] = filename
                all_chunks.extend(self.chunk_text(body, meta))

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(all_chunks, f, indent=4)

        return len(all_chunks)

if __name__ == '__main__':
    ingestor = BankGPTIngestor()
    count = ingestor.run('data/raw', 'data/processed/chunks.json')
    print(f'Successfully created {count} chunks in data/processed/chunks.json')
"""

with open('src/ingestion.py', 'w') as f:
    f.write(ingestion_script)

!python src/ingestion.py

Successfully created 3 chunks in data/processed/chunks.json


## Phase 4: Dense & Sparse Indexing (FAISS & BM25)

### Subtask:
Create the `src/indexer.py` module to build and save the hybrid search indices.

In [ ]:
# Debugging the JSON output and re-running the indexer
import json

try:
    with open('data/processed/chunks.json', 'r') as f:
        data = json.load(f)
    print(f'JSON is valid. Number of chunks: {len(data)}')
    # Re-running the indexing script
    !python src/indexer.py
except Exception as e:
    print(f'Error detected in chunks.json: {e}')
    # Display the first few characters of the file for debugging
    with open('data/processed/chunks.json', 'r') as f:
        print('File snippet:', f.read(500))

JSON is valid. Number of chunks: 3
Loading embedding model: BAAI/bge-small-en-v1.5...
Loading weights: 100% 199/199 [00:00<00:00, 4354.93it/s]
Generating embeddings for FAISS...
Batches: 100% 1/1 [00:00<00:00,  1.50it/s]
Building BM25 index...
Indices saved successfully.


## Phase 5: Hybrid Retrieval & Reranking

### Subtask:
Implement `src/retriever.py` to merge dense/sparse results and apply a Cross-Encoder reranker.

In [ ]:
retriever_script = """
import json
import pickle
import faiss
import numpy as np
import os
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

# Suppress HF authentication warnings in logs
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

class HybridRetriever:
    def __init__(self, faiss_path, bm25_path, chunks_path, model_name='BAAI/bge-small-en-v1.5'):
        print("Initializing Hybrid Retriever with RRF...")
        self.embed_model = SentenceTransformer(model_name)
        self.rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

        self.faiss_index = faiss.read_index(faiss_path)
        with open(bm25_path, 'rb') as f:
            self.bm25 = pickle.load(f)
        with open(chunks_path, 'r') as f:
            self.chunks = json.load(f)

    def search(self, query, top_k=5, rrf_k=60):
        # 1. Dense Search
        query_emb = self.embed_model.encode([query])
        faiss.normalize_L2(query_emb)
        dense_scores, dense_indices = self.faiss_index.search(query_emb, top_k * 4)

        # 2. Sparse Search
        tokenized_query = query.lower().split()
        sparse_scores = self.bm25.get_scores(tokenized_query)
        sparse_indices = np.argsort(sparse_scores)[::-1][:top_k * 4]

        # 3. Reciprocal Rank Fusion (RRF)
        rrf_scores = {}

        # Dense Rank scores
        for rank, idx in enumerate(dense_indices[0]):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (rrf_k + rank)

        # Sparse Rank scores
        for rank, idx in enumerate(sparse_indices):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (rrf_k + rank)

        # Sort by RRF score and pick top candidates for reranking
        sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_k * 3]
        candidates = [self.chunks[i]['text_content'] for i in sorted_indices]

        # 4. Cross-Encoder Reranking
        model_inputs = [[query, c] for c in candidates]
        rerank_scores = self.rerank_model.predict(model_inputs)

        # Final Sort
        results = sorted(zip(sorted_indices, rerank_scores), key=lambda x: x[1], reverse=True)

        final_chunks = []
        for idx, score in results[:top_k]:
            chunk = self.chunks[idx].copy()
            chunk['rerank_score'] = float(score)
            final_chunks.append(chunk)

        return final_chunks

if __name__ == '__main__':
    retriever = HybridRetriever(
        'models/embeddings/faiss_index.bin',
        'models/embeddings/bm25_index.pkl',
        'data/processed/chunks.json'
    )
    test_query = "How has the RBI handled the repo rate and what is the target inflation?"
    results = retriever.search(test_query)

    print(f"\\nQuery: {test_query}")
    print("-" * 30)
    for i, res in enumerate(results):
        print(f"Rank {i+1} (Score: {res['rerank_score']:.4f}):")
        print(f"{res['text_content'][:200]}...\\n")
"""

with open('src/retriever.py', 'w') as f:
    f.write(retriever_script)

# Run the updated retriever
!python src/retriever.py

Initializing Hybrid Retriever with RRF...
Loading weights: 100% 199/199 [00:00<00:00, 5440.12it/s]
Loading weights: 100% 105/105 [00:00<00:00, 13347.54it/s]

Query: How has the RBI handled the repo rate and what is the target inflation?
------------------------------
Rank 1 (Score: 4.8753):
Source: RBI Policy Update March 2026 | Bank: General | Date: 2026-03-01 | # RBI Holds Repo Rate at 6.25%

In the final MPC meeting of FY 2025-26, the Reserve Bank of India decided to keep the repo rat...

Rank 2 (Score: 4.8753):
Source: RBI Policy Update March 2026 | Bank: General | Date: 2026-03-01 | # RBI Holds Repo Rate at 6.25%

In the final MPC meeting of FY 2025-26, the Reserve Bank of India decided to keep the repo rat...

Rank 3 (Score: -10.9196):
Source: Indian Banking Glossary 2025 | Bank: General | Date: 2025-01-10 | # Indian Banking Glossary

- **Unified Ledger Interface (ULI)**: A cross-institutional platform for seamless credit assessment...

Rank 4 (Score: -11.3728):
Source: Vikas Ban

## Phase 6: Quantized Inference Engine (Llama 3.2)

### Subtask:
Implement `src/generator.py` to orchestrate the RAG pipeline using a 4-bit quantized Llama 3.2 model.

In [4]:
import os

# Ensure src directory exists and is a package
os.makedirs('src', exist_ok=True)
with open('src/__init__.py', 'a') as f:
    pass

llm_script = """
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from src.retriever import HybridRetriever
import os

class BankGPTGenerator:
    def __init__(self, model_id='unsloth/Llama-3.2-3B-Instruct-bnb-4bit'):
        print(f'Loading Quantized LLM: {model_id}...')

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map='auto',
            trust_remote_code=True
        )

        self.retriever = HybridRetriever(
            'models/embeddings/faiss_index.bin',
            'models/embeddings/bm25_index.pkl',
            'data/processed/chunks.json'
        )

    def format_prompt(self, query, context):
        prompt = f\"\"\"<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are BankGPT, a professional AI assistant for the Indian Banking Sector (2025-2026).
Use the provided CONTEXT to answer the User Query.

### CONTEXT:
{context}
<|eot_id|><|start_header_id|>user<|end_header_id|>

{query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\"\"\"
        return prompt

    def generate_response(self, query, max_new_tokens=512):
        search_results = self.retriever.search(query, top_k=3)
        context_text = '\\n\\n'.join([r['text_content'] for r in search_results])

        prompt = self.format_prompt(query, context_text)
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id
        )

        full_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = full_text.split('assistant')[-1].strip()
        return response, search_results

if __name__ == '__main__':
    bot = BankGPTGenerator()
    test_query = 'What is the current repo rate according to RBI March 2026 news?'
    ans, sources = bot.generate_response(test_query)
    print('='*40)
    print(f'BankGPT: {ans}')
"""

with open('src/llm.py', 'w') as f:
    f.write(llm_script)

print('Phase 6 LLM module created at src/llm.py')

Phase 6 LLM module created at src/llm.py


In [30]:
import os

# 1. Ensure package structure
for p in ['src', 'src/retrieval']:
    os.makedirs(p, exist_ok=True)
    with open(os.path.join(p, '__init__.py'), 'a') as f: pass

# 2. Explicitly create the retriever module in the correct package location
retriever_code = """
import json
import pickle
import faiss
import numpy as np
import os
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

class HybridRetriever:
    def __init__(self, faiss_path, bm25_path, chunks_path, model_name='BAAI/bge-small-en-v1.5'):
        self.embed_model = SentenceTransformer(model_name)
        self.rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        self.faiss_index = faiss.read_index(faiss_path)
        with open(bm25_path, 'rb') as f:
            self.bm25 = pickle.load(f)
        with open(chunks_path, 'r') as f:
            self.chunks = json.load(f)

    def search(self, query, top_k=5, rrf_k=60):
        query_emb = self.embed_model.encode([query])
        faiss.normalize_L2(query_emb)
        dense_scores, dense_indices = self.faiss_index.search(query_emb, top_k * 4)
        tokenized_query = query.lower().split()
        sparse_scores = self.bm25.get_scores(tokenized_query)
        sparse_indices = np.argsort(sparse_scores)[::-1][:top_k * 4]
        rrf_scores = {}
        for rank, idx in enumerate(dense_indices[0]):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (rrf_k + rank)
        for rank, idx in enumerate(sparse_indices):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (rrf_k + rank)
        sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_k * 3]
        candidates = [self.chunks[i]['text_content'] for i in sorted_indices]
        model_inputs = [[query, c] for c in candidates]
        rerank_scores = self.rerank_model.predict(model_inputs)
        results = sorted(zip(sorted_indices, rerank_scores), key=lambda x: x[1], reverse=True)
        final_chunks = []
        for idx, score in results[:top_k]:
            chunk = self.chunks[idx].copy()
            chunk['rerank_score'] = float(score)
            final_chunks.append(chunk)
        return final_chunks
"""

with open('src/retrieval/retriever.py', 'w') as f:
    f.write(retriever_code)

print('Retriever module placed in src/retrieval/retriever.py successfully.')

Retriever module placed in src/retrieval/retriever.py successfully.


In [34]:
# 1. Ensure all retrieval dependencies are present
!pip install -q faiss-cpu rank_bm25

import sys
import os
import torch

# 2. Setup paths
PROJECT_ROOT = '/content'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.retrieval.retriever import HybridRetriever
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

class BankGPTGenerator:
    def __init__(self, model_id='unsloth/Llama-3.2-3B-Instruct-bnb-4bit'):
        print(f'Loading Quantized LLM: {model_id}...')

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map='auto',
            trust_remote_code=True
        )

        self.retriever = HybridRetriever(
            'models/embeddings/faiss_index.bin',
            'models/embeddings/bm25_index.pkl',
            'data/processed/chunks.json'
        )

    def format_prompt(self, query, context):
        return f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are BankGPT, specialized in Indian Banking (2025-2026).\n\n### CONTEXT:\n{context}\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"

    def generate_response(self, query):
        print("Retrieving context...")
        search_results = self.retriever.search(query, top_k=3)
        context_text = "\n\n".join([r['text_content'] for r in search_results])

        print("Generating answer...")
        prompt = self.format_prompt(query, context_text)
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id
        )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True).split('assistant')[-1].strip()

# 3. Execution
bot = BankGPTGenerator()
test_query = 'What is the current repo rate and how is ULI adoption progressing?'
response = bot.generate_response(test_query)

print(f'\nQuery: {test_query}')
print('='*40)
print(f'BankGPT: {response}')

Loading Quantized LLM: unsloth/Llama-3.2-3B-Instruct-bnb-4bit...


config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:271: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [48]:
# 1. Force CUDA detection and backend refresh
import os
import sys
import torch
import importlib

# Ensure CUDA libraries are in the dynamic linker path
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')

# Force refresh of bitsandbytes and its internal state
import bitsandbytes as bnb
importlib.reload(bnb)

print(f"Verified bitsandbytes version: {bnb.__version__}")

# 2. Re-import RAG components
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from src.retrieval.retriever import HybridRetriever

# 3. Define and Run the Generator
try:
    class BankGPTGenerator:
        def __init__(self, model_id='unsloth/Llama-3.2-3B-Instruct-bnb-4bit'):
            print(f'Loading Quantized LLM: {model_id}...')

            # Use a fresh config to override model defaults
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.bfloat16
            )

            self.tokenizer = AutoTokenizer.from_pretrained(model_id)
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=bnb_config,
                device_map='auto',
                trust_remote_code=True
            )

            self.retriever = HybridRetriever(
                'models/embeddings/faiss_index.bin',
                'models/embeddings/bm25_index.pkl',
                'data/processed/chunks.json'
            )

        def generate_response(self, query):
            search_results = self.retriever.search(query, top_k=3)
            context_text = '\n\n'.join([r['text_content'] for r in search_results])

            prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are BankGPT. Context:\n{context_text}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"

            inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.1,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )

            response = self.tokenizer.decode(outputs[0], skip_special_tokens=True).split('assistant')[-1].strip()
            return response

    bot = BankGPTGenerator()
    test_query = 'What is the current repo rate and how is ULI adoption progressing?'

    print(f'\n--- Phase 6: Final Inference Test ---')
    response = bot.generate_response(test_query)

    print(f'\nQuery: {test_query}')
    print('='*60)
    print(f'BankGPT Response:\n{response}')
    print('='*60)

except Exception as e:
    print(f'Pipeline Error: {e}')

Verified bitsandbytes version: 0.49.2
Loading Quantized LLM: unsloth/Llama-3.2-3B-Instruct-bnb-4bit...
Pipeline Error: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:271: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


### Phase 6: Final Inference Integration & Verification
This step involves loading the 4-bit quantized Llama 3.2 model and executing the full RAG pipeline.

In [5]:
import os, sys

PROJECT_ROOT = "/content/bankgpt"  # change to your actual root
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Persist across sub-shells (e.g. if you shell out with !python later)
os.environ["PYTHONPATH"] = PROJECT_ROOT + ":" + os.environ.get("PYTHONPATH", "")

print(sys.path[:3])

Current working directory: /content
Python Path: ['/content/bankgpt', '/content', '/env/python']


In [6]:
!pip install -q -U bitsandbytes accelerate

import os
# T4 runtimes sometimes don't expose the nvidia libs on the default linker path
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")

import torch
print("CUDA available:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0))

CUDA available: True | Device: Tesla T4


In [9]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_Token')
login(token=hf_token)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [11]:
import torch

# 1. Check the primary device
print(f"Model is primarily on: {model.device}")

# 2. Check quantization status via config
if hasattr(model, 'config') and hasattr(model.config, 'quantization_config'):
    print("Quantization Config:", model.config.quantization_config.to_dict())
else:
    print("Quantization config not found.")

# 3. Check GPU memory usage
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# 4. Total parameter check
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params / 1e9:.2f}B")

Model is primarily on: cuda:0
Quantization Config: {'quant_method': <QuantizationMethod.BITS_AND_BYTES: 'bitsandbytes'>, '_load_in_8bit': False, '_load_in_4bit': True, 'llm_int8_threshold': 6.0, 'llm_int8_skip_modules': None, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_storage': 'uint8', 'load_in_4bit': True, 'load_in_8bit': False}
GPU memory allocated: 2.24 GB
GPU memory reserved: 2.29 GB
Total parameters: 1.80B


In [13]:
print("Model name/path:", model.config._name_or_path)
print("Config num_hidden_layers:", model.config.num_hidden_layers)
print("Config hidden_size:", model.config.hidden_size)

Model name/path: meta-llama/Llama-3.2-3B-Instruct
Config num_hidden_layers: 28
Config hidden_size: 3072


In [16]:
import torch

# --- Part 1: Confirm identity and placement ---
print(f"Model name/path: {model.config._name_or_path}")
print(f"Num hidden layers: {model.config.num_hidden_layers}")  # 28 for 3B
print(f"Hidden size: {model.config.hidden_size}")               # 3072 for 3B

# Safely check for device placement
print(f"Model primary device: {model.device}")

# Check quantization status
is_4bit = getattr(model, 'is_loaded_in_4bit', False)
print(f"Is loaded in 4-bit: {is_4bit}")

if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("\n" + "="*50 + "\n")

# --- Part 2: Actual generation test ---
test_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "In one sentence, what is the RBI?"}
]

# apply_chat_template returns a BatchEncoding object
inputs = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(model.device)

with torch.no_grad():
    output = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Slice the output to show only the new tokens
response = tokenizer.decode(output[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
print("Test prompt: In one sentence, what is the RBI?")
print("Model response:", response)

Model name/path: meta-llama/Llama-3.2-3B-Instruct
Num hidden layers: 28
Hidden size: 3072
Model primary device: cuda:0
Is loaded in 4-bit: True
GPU memory allocated: 2.24 GB




/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Test prompt: In one sentence, what is the RBI?
Model response: The RBI stands for the Reserve Bank of India, which is the central bank of India and the country's primary monetary authority, responsible for regulating the nation's financial system and maintaining its economic stability.


In [21]:
import sys, os

# 1. Ensure absolute paths and package markers
for path in ['src', 'src/retrieval']:
    os.makedirs(path, exist_ok=True)
    init_file = os.path.join(path, '__init__.py')
    if not os.path.exists(init_file):
        with open(init_file, 'w') as f: pass

# 2. Re-write retriever.py explicitly to ensure it exists
retriever_code = """
import json
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

class HybridRetriever:
    def __init__(self, faiss_path, bm25_path, chunks_path, model_name='BAAI/bge-small-en-v1.5'):
        self.embed_model = SentenceTransformer(model_name)
        self.rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        self.faiss_index = faiss.read_index(faiss_path)
        with open(bm25_path, 'rb') as f: self.bm25 = pickle.load(f)
        with open(chunks_path, 'r') as f: self.chunks = json.load(f)

    def search(self, query, top_k=3):
        query_emb = self.embed_model.encode([query])
        faiss.normalize_L2(query_emb)
        _, dense_indices = self.faiss_index.search(query_emb, top_k * 4)
        tokenized_query = query.lower().split()
        sparse_scores = self.bm25.get_scores(tokenized_query)
        sparse_indices = np.argsort(sparse_scores)[::-1][:top_k * 4]
        combined_indices = list(set(dense_indices[0]) | set(sparse_indices))
        candidates = [self.chunks[i]['text_content'] for i in combined_indices]
        rerank_scores = self.rerank_model.predict([[query, c] for c in candidates])
        results = sorted(zip(combined_indices, rerank_scores), key=lambda x: x[1], reverse=True)
        return [self.chunks[idx] for idx, _ in results[:top_k]]
"""

with open('src/retrieval/retriever.py', 'w') as f:
    f.write(retriever_code)

# 3. Add /content to sys.path
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# 4. Import and Execute
try:
    from src.retrieval.retriever import HybridRetriever
    import torch

    retriever = HybridRetriever(
        faiss_path='models/embeddings/faiss_index.bin',
        bm25_path='models/embeddings/bm25_index.pkl',
        chunks_path='data/processed/chunks.json'
    )

    def ask_bankgpt(query):
        results = retriever.search(query, top_k=3)
        context = "\n\n".join([r['text_content'] for r in results])

        messages = [
            {"role": "system", "content": "You are BankGPT. Use the provided context to answer the user's banking query accurately."},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUERY: {query}"}
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        response = tokenizer.decode(output[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip()

    test_query = "What is the current repo rate and how is ULI adoption progressing?"
    print(f"User Query: {test_query}")
    print("-" * 50)
    print("BankGPT Answer:")
    print(ask_bankgpt(test_query))

except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

An error occurred: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/faiss/impl/io.cpp:69: Error: 'f' failed: could not open models/embeddings/faiss_index.bin for reading: No such file or directory


Traceback (most recent call last):
  File "/tmp/ipykernel_779/2935990667.py", line 53, in <cell line: 0>
    retriever = HybridRetriever(
                ^^^^^^^^^^^^^^^^
  File "/content/src/retrieval/retriever.py", line 13, in __init__
    self.faiss_index = faiss.read_index(faiss_path)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/faiss/swigfaiss.py", line 13591, in read_index
    return _swigfaiss.read_index(*args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/faiss/impl/io.cpp:69: Error: 'f' failed: could not open models/embeddings/faiss_index.bin for reading: No such file or directory


## Phase 7: Multi-Hop Agent Development

### Subtask:
Implement a reasoning agent that can decompose complex banking queries (e.g., comparing two banks' performance) into sequential retrieval steps.

In [43]:
agent_script = """
import torch
import json
import re

class BankGPTAgent:
    def __init__(self, generator):
        self.generator = generator

    def decompose_query(self, query):
        # Refined prompt for stricter JSON output without preamble
        prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\\n\\nYou are a banking research assistant. Break the user query into exactly two independent search strings. Output ONLY a JSON list of strings. No conversational text.\\n\\nExample: 'Compare X profit and Y NPA' -> [\\\"net profit of X bank\\\", \\\"net NPA of Y bank\\\"]\\n\\nQUERY: {query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"

        inputs = self.generator.tokenizer(prompt, return_tensors='pt').to(self.generator.model.device)
        outputs = self.generator.model.generate(**inputs, max_new_tokens=60, do_sample=False)
        response = self.generator.tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()

        try:
            # Extract JSON array using regex
            match = re.search(r'\\[.*\\]', response, re.DOTALL)
            if match:
                return json.loads(match.group())
            return [query]
        except:
            return [query]

    def run(self, query):
        sub_queries = self.decompose_query(query)
        print(f"Agent Decomposed Tasks: {sub_queries}")

        context_parts = []
        for sub in sub_queries:
            print(f"Searching knowledge base for: {sub}")
            results = self.generator.retriever.search(sub, top_k=2)
            context_parts.append("\\n".join([r['text_content'] for r in results]))

        final_context = "\\n\\n---\\n\\n".join(context_parts)
        return self.generator.ask_bankgpt_with_context(query, final_context)
"""

with open('src/agent.py', 'w') as f:
    f.write(agent_script)

print('Multi-hop logic updated to handle sub-query decomposition and context merging.')

Multi-hop logic updated to handle sub-query decomposition and context merging.


In [28]:
import torch
import sys
import os
import pickle
import json

# 1. Re-create the indexer.py script
indexer_script = """
import json
import pickle
import faiss
import os
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

def build_indices(chunks_path, faiss_out, bm25_out):
    with open(chunks_path, 'r') as f:
        chunks = json.load(f)
    texts = [c['text_content'] for c in chunks]

    # Dense Index
    model = SentenceTransformer('BAAI/bge-small-en-v1.5')
    embeddings = model.encode(texts, show_progress_bar=True)
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    faiss.write_index(index, faiss_out)

    # Sparse Index
    tokenized_corpus = [doc.lower().split() for doc in texts]
    bm25 = BM25Okapi(tokenized_corpus)
    with open(bm25_out, 'wb') as f:
        pickle.dump(bm25, f)

if __name__ == '__main__':
    os.makedirs('models/embeddings', exist_ok=True)
    build_indices('data/processed/chunks.json', 'models/embeddings/faiss_index.bin', 'models/embeddings/bm25_index.pkl')
    print('Indices built successfully.')
"""

with open('src/indexer.py', 'w') as f:
    f.write(indexer_script)

# 2. Run Ingestion and Indexing to ensure files exist
!python src/ingestion.py
!python src/indexer.py

# 3. Setup paths and imports
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
from src.retrieval.retriever import HybridRetriever

retriever = HybridRetriever(
    faiss_path='models/embeddings/faiss_index.bin',
    bm25_path='models/embeddings/bm25_index.pkl',
    chunks_path='data/processed/chunks.json'
)

# 4. Helper and Agent Initialization
class MultiHopWrapper:
    def __init__(self, model, tokenizer, retriever):
        self.model = model
        self.tokenizer = tokenizer
        self.retriever = retriever

    def ask_bankgpt_with_context(self, query, context):
        messages = [
            {"role": "system", "content": "You are BankGPT. Use the provided context to answer accurately."},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUERY: {query}"}
        ]
        inputs = self.tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output = self.model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True, pad_token_id=self.tokenizer.eos_token_id)
        return self.tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)

import importlib
import src.agent
importlib.reload(src.agent)
from src.agent import BankGPTAgent

agent = BankGPTAgent(MultiHopWrapper(model, tokenizer, retriever))

complex_query = "Compare the NPA of Vikas Bank with the profit highlights of Apex National Bank."
print(f"Processing Multi-Hop Query: {complex_query}")
print("-" * 50)
print(agent.run(complex_query))

python3: can't open file '/content/src/ingestion.py': [Errno 2] No such file or directory
Traceback (most recent call last):
  File "/content/src/indexer.py", line 30, in <module>
    build_indices('data/processed/chunks.json', 'models/embeddings/faiss_index.bin', 'models/embeddings/bm25_index.pkl')
  File "/content/src/indexer.py", line 10, in build_indices
    with open(chunks_path, 'r') as f:
         ^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/chunks.json'


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RuntimeError: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/faiss/impl/io.cpp:69: Error: 'f' failed: could not open models/embeddings/faiss_index.bin for reading: No such file or directory

In [29]:
import os
import sys

# Reset working directory to /content
os.chdir('/content')
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# Ensure directory structure exists
for d in ['data/raw', 'data/processed', 'models/embeddings', 'src/retrieval']:
    os.makedirs(d, exist_ok=True)
    with open(os.path.join(d.split('/')[0], '__init__.py'), 'a') as f: pass

print('Directory structure verified.')

Directory structure verified.


In [37]:
indexer_script = """
import json, pickle, faiss, os
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

def build_indices(chunks_path, faiss_out, bm25_out):
    if not os.path.exists(chunks_path):
        return print(f'Error: {chunks_path} not found.')
    with open(chunks_path, 'r') as f: chunks = json.load(f)
    if not chunks: return print('No chunks found to index.')

    texts = [c['text_content'] for c in chunks]
    model = SentenceTransformer('BAAI/bge-small-en-v1.5')
    embeddings = model.encode(texts, show_progress_bar=True)

    if len(embeddings.shape) == 1:
        embeddings = embeddings.reshape(1, -1)

    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    faiss.write_index(index, faiss_out)

    tokenized_corpus = [doc.lower().split() for doc in texts]
    bm25 = BM25Okapi(tokenized_corpus)
    with open(bm25_out, 'wb') as f:
        pickle.dump(bm25, f)
    print('Indices built successfully.')

if __name__ == '__main__':
    os.makedirs('models/embeddings', exist_ok=True)
    build_indices('data/processed/chunks.json', 'models/embeddings/faiss_index.bin', 'models/embeddings/bm25_index.pkl')
"""
with open('src/indexer.py', 'w') as f: f.write(indexer_script)

# Run Ingestion
ingestor = BankGPTIngestor()
ingestor.run('data/raw', 'data/processed/chunks.json')

# Run Indexing
!python src/indexer.py
print('Phase 4: Indexing confirmed.')

Loading weights: 100% 199/199 [00:00<00:00, 7030.72it/s]
Batches: 100% 1/1 [00:00<00:00,  2.49it/s]
Indices built successfully.
Phase 4: Indexing confirmed.


In [44]:
import importlib
import src.agent
from src.retrieval.retriever import HybridRetriever
import torch

# Re-initialize retriever and agent to apply logic updates
retriever = HybridRetriever(
    faiss_path='models/embeddings/faiss_index.bin',
    bm25_path='models/embeddings/bm25_index.pkl',
    chunks_path='data/processed/chunks.json'
)

importlib.reload(src.agent)
from src.agent import BankGPTAgent

agent = BankGPTAgent(MultiHopWrapper(model, tokenizer, retriever))

complex_query = "Compare the NPA of Vikas Bank with the profit highlights of Apex National Bank."
print(f"Processing: {complex_query}")
print("=" * 50)
result = agent.run(complex_query)
print("\\nBankGPT Final Answer:")
print(result)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Processing: Compare the NPA of Vikas Bank with the profit highlights of Apex National Bank.


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Agent Decomposed Tasks: ['net NPA of Vikas Bank', 'profit highlights of Apex National Bank']
Searching knowledge base for: net NPA of Vikas Bank
Searching knowledge base for: profit highlights of Apex National Bank
\nBankGPT Final Answer:
To compare the NPA of Vikas Bank with the profit highlights of Apex National Bank, we need to analyze the information provided.

Vikas Bank's Net NPA has dropped to 2.1% in Q1 FY25-26.

Apex National Bank reported a Net Profit of ₹12,500 crores.

There is no direct comparison between the NPA of Vikas Bank and the profit highlights of Apex National Bank. NPA (Net Non-Performing Assets) is a measure of the amount of loans that are not being repaid by borrowers, whereas Net Profit is a measure of the bank's overall financial performance.

However, we can make a general observation that a lower NPA ratio (2.1% in this case) is generally considered a positive indicator of a bank's credit risk management and overall financial health. On the other hand, a hi

## Phase 8: Performance Evaluation

### Subtask:
Implement a fact-checking and relevance evaluation suite to measure the accuracy of BankGPT responses against the ground-truth synthetic data.

In [47]:
evaluation_script = """
import torch
import re

class RAGEvaluator:
    def __init__(self, generator):
        self.generator = generator

    def evaluate_faithfulness(self, query, response, context):
        # Correctly formatted template to avoid SyntaxErrors in f-strings
        prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\\n\\nYou are an AI Auditor. Evaluate if the ANSWER is supported ONLY by the CONTEXT. Provide a score from 0 to 10. Output ONLY the numeric score.\\n\\nCONTEXT: {context}\\n\\nANSWER: {response}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"

        inputs = self.generator.tokenizer(prompt, return_tensors='pt').to(self.generator.model.device)
        outputs = self.generator.model.generate(**inputs, max_new_tokens=10, do_sample=False)
        score_str = self.generator.tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()

        try:
            nums = re.findall(r'\\d+', score_str)
            return int(nums[0]) if nums else 5
        except:
            return 5

if __name__ == '__main__':
    print('Evaluator module defined.')
"""

with open('src/evaluation.py', 'w') as f:
    f.write(evaluation_script)

print('Phase 8 evaluation logic corrected and saved to src/evaluation.py')

Phase 8 evaluation logic corrected and saved to src/evaluation.py


In [48]:
import importlib
import src.evaluation
importlib.reload(src.evaluation)
from src.evaluation import RAGEvaluator

evaluator = RAGEvaluator(agent.generator)

# Reference content for verification
ground_truth = "Vikas Bank NPA is 2.1%. Apex National Bank reported profit of ₹12,500 crores."

faithfulness_score = evaluator.evaluate_faithfulness(complex_query, result, ground_truth)

print(f"Evaluation for Query: {complex_query}")
print(f"Faithfulness Score (0-10): {faithfulness_score}")

Evaluation for Query: Compare the NPA of Vikas Bank with the profit highlights of Apex National Bank.
Faithfulness Score (0-10): 6


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


## Phase 9: Gradio UI Deployment

### Subtask:
Create an interactive dashboard for BankGPT to allow real-time multi-hop queries and view retrieval reasoning.

In [50]:
import gradio as gr

def predict(message, history):
    # The agent.run method handles the multi-hop decomposition and final answering
    try:
        response = agent.run(message)
        return response
    except Exception as e:
        return f"An error occurred during agent execution: {str(e)}"

# Launching a persistent interface
demo = gr.ChatInterface(
    fn=predict,
    title="BankGPT: Indian Banking Intelligence (FY 2025-26)",
    description="Interactive Multi-Hop RAG Agent for analyzing financial reports and RBI policies.",
    examples=[
        "Compare the NPA of Vikas Bank with the profit highlights of Apex National Bank.",
        "What are the latest RBI guidelines on Penal Charges for 2026?",
        "Explain the impact of ULI on personal loan approvals."
    ],
    theme="soft"
)

# Set share=True to generate a public URL
demo.launch(share=True, debug=False)

TypeError: ChatInterface.__init__() got an unexpected keyword argument 'theme'

## Phase 9: Gradio UI Deployment

### Subtask:
Create an interactive dashboard for BankGPT to allow real-time multi-hop queries and view retrieval sources.

In [51]:
import gradio as gr

def predict(message, history):
    # The agent.run method handles the multi-hop logic
    response = agent.run(message)
    return response

# Launching a simple but functional interface
demo = gr.ChatInterface(
    fn=predict,
    title="BankGPT: Indian Banking Intelligence (FY 2025-26)",
    description="Ask complex questions about Indian banks, RBI policies, and financial metrics.",
    examples=["Compare Vikas Bank NPA with Apex National Bank profit.", "What are the RBI rules for EBLR?"]
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a6b651365fe3712ac1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
